!pip install requests beautifulsoup4 pandas numpy matplotlib seaborn scipy nltk wordcloud -q

In [1]:
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

print('All libraries ready!')

All libraries ready!


# TASK 1: Web Scraping

In [2]:
#Goal: Scrape book listings from `books.toscrape.com` — a free, legal scraping practice site.
#A synthetic dataset is automatically generated as fallback if the site is unreachable.

#Tools: `requests`, `BeautifulSoup`

In [3]:
# Import required libraries
# requests  : used to send HTTP requests to the website
# BeautifulSoup : used to parse and extract HTML content
# pandas    : used for data storage and manipulation
# numpy     : used for generating synthetic (random) data
# time      : used to add delay between requests (ethical scraping)

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time

# Dictionary to convert textual star ratings into numeric values
# Example: "Three" → 3
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# HTTP headers with a User-Agent to identify the request
# This helps avoid blocking and follows ethical scraping practices
HEADERS = {
    "User-Agent": "Mozilla/5.0 (educational scraping project)"
}


# Function to scrape book data from books.toscrape.com
def scrape_books(pages=5):
    """
    Scrapes book information from multiple pages of books.toscrape.com.

    Parameters:
    pages (int): Number of pages to scrape

    Returns:
    DataFrame: Pandas DataFrame containing scraped book data
    """

    books_data = []  # List to store scraped book information

    # Loop through each page
    for page_num in range(1, pages + 1):

        # Construct page URL dynamically
        url = f"http://books.toscrape.com/catalogue/page-{page_num}.html"

        # Send GET request to the website
        response = requests.get(url, headers=HEADERS, timeout=10)

        # Raise an error if the request fails (e.g., 404 or 500)
        response.raise_for_status()

        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.text, "html.parser")

        # Select all book containers on the page
        for book in soup.select("article.product_pod"):

            # Extract book title
            title = book.h3.a["title"]

            # Extract and clean book price (remove currency symbol)
            price_text = book.select_one(".price_color").text.strip()
            price = float(''.join(c for c in price_text if c.isdigit() or c == '.'))

            # Extract star rating and convert it to numeric form
            rating_text = book.p["class"][1]
            rating = RATING_MAP.get(rating_text, 0)

            # Extract availability status
            availability = book.select_one(".availability").text.strip()

            # Store extracted data in dictionary format
            books_data.append({
                "Title": title,
                "Price_GBP": price,
                "Rating": rating,
                "Availability": availability,
                "Page": page_num
            })

        # Print progress message for each page
        print(f"Page {page_num} scraped successfully")

        # Delay between requests to avoid overloading the server
        time.sleep(0.5)

    # Convert the collected data into a Pandas DataFrame
    return pd.DataFrame(books_data)


# Function to generate a synthetic dataset if live scraping fails
def generate_synthetic_books(n=100):
    """
    Generates a synthetic book dataset for offline use or testing.

    Parameters:
    n (int): Number of synthetic records to generate

    Returns:
    DataFrame: Pandas DataFrame with generated book data
    """

    # Set random seed for reproducibility
    np.random.seed(42)

    # Sample genres and authors
    genres = [
        "Fiction", "Mystery", "Sci-Fi", "Romance", "History",
        "Biography", "Self-Help", "Thriller", "Fantasy", "Horror"
    ]

    authors = [
        "Alice Hartman", "Bob Chen", "Clara Voss", "David Kim",
        "Eva Martinez", "Frank O'Neill", "Grace Lee", "Henry Park"
    ]

    # Randomly generate data for each column
    genre_col = np.random.choice(genres, n)
    titles = [f"The {g} Chronicles Vol.{i+1}" for i, g in enumerate(genre_col)]
    ratings = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.10, 0.20, 0.35, 0.30])
    prices = np.round(np.random.uniform(5, 55, n), 2)
    availability = np.random.choice(["In stock", "Out of stock"], size=n, p=[0.85, 0.15])

    # Create and return the DataFrame
    return pd.DataFrame({
        "Title": titles,
        "Price_GBP": prices,
        "Rating": ratings,
        "Availability": availability,
        "Genre": genre_col,
        "Author": np.random.choice(authors, n),
        "Page": np.random.randint(1, 6, n)
    })


# Main execution block
print("Attempting live web scrape from books.toscrape.com...")

try:
    # Try to scrape live data
    df_books = scrape_books(pages=5)
    print(f"\nLive scraping SUCCESS - {len(df_books)} books collected")

except Exception as e:
    # If scraping fails, generate synthetic data instead
    print(f"Live scraping unavailable ({type(e).__name__}). Using synthetic dataset...")
    df_books = generate_synthetic_books(n=100)
    print(f"Synthetic dataset generated - {len(df_books)} books")

# Save the final dataset to a CSV file
df_books.to_csv("scraped_books.csv", index=False)

# Display dataset information
print(f"\nDataset saved: scraped_books.csv | Shape: {df_books.shape}")

# Display first 8 records for verification
df_books.head(8)

Attempting live web scrape from books.toscrape.com...
Page 1 scraped successfully
Page 2 scraped successfully
Page 3 scraped successfully
Page 4 scraped successfully
Page 5 scraped successfully

Live scraping SUCCESS - 100 books collected

Dataset saved: scraped_books.csv | Shape: (100, 5)


,Title,Price_GBP,Rating,Availability,Page
0,A Light in the Attic,51.77,3,In stock,1
1,Tipping the Velvet,53.74,1,In stock,1
2,Soumission,50.10,1,In stock,1
3,Sharp Objects,47.82,4,In stock,1
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,1
5,The Requiem Red,22.65,1,In stock,1
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,In stock,1
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,In stock,1


In [4]:
# Display a heading for the dataset summary
print("Dataset Quick Summary")

# Print the list of column names in the DataFrame
print(f"Columns    : {list(df_books.columns)}")

# Print the total number of book records in the dataset
print(f"Total Books: {len(df_books)}")

# Print the minimum and maximum book prices in GBP (formatted to 2 decimal places)
print(
    f"Price Range: GBP {df_books['Price_GBP'].min():.2f} "
    f"to {df_books['Price_GBP'].max():.2f}"
)

# Print the average price of all books
print(f"Avg Price  : GBP {df_books['Price_GBP'].mean():.2f}")

# Print the average rating of books out of 5
print(f"Avg Rating : {df_books['Rating'].mean():.2f} / 5")

# Print a heading for rating distribution
print("\nRating Distribution:")

# Count how many books fall into each rating category
# sort_index() ensures ratings are shown in ascending order (1 to 5)
print(df_books['Rating'].value_counts().sort_index())

Dataset Quick Summary
Columns    : ['Title', 'Price_GBP', 'Rating', 'Availability', 'Page']
Total Books: 100
Price Range: GBP 10.16 to 58.11
Avg Price  : GBP 34.56
Avg Rating : 2.93 / 5

Rating Distribution:
Rating
1    22
2    19
3    22
4    18
5    19
Name: count, dtype: int64


In [5]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("scraped_books.csv")

print("=" * 60)
print("        EXPLORATORY DATA ANALYSIS REPORT")
print("=" * 60)
print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
missing = df.isnull().sum()
if missing.any():
    print(missing[missing > 0])
else:
    print("  None found")
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

        EXPLORATORY DATA ANALYSIS REPORT

Shape: 100 rows x 5 columns

Data Types:
Title            object
Price_GBP       float64
Rating            int64
Availability     object
Page              int64
dtype: object

Missing Values:
  None found

Duplicate Rows: 0
